In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-latest",
    temperature = 0.8
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical writer."),
    ("human", "Explain {topic} in 2 sentences.")
])

# the most basic output parser in StrOutputParser which returns a str.
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

chain.invoke({
    "topic" : "Object Oriented Programming"
})

'Object-Oriented Programming (OOP) is a software design paradigm that structures program code around data, or "objects," rather than functions and logic. By leveraging the core principles of encapsulation, inheritance, polymorphism, and abstraction, OOP enables developers to build highly reusable, modular, and scalable systems.'

In [3]:
# sometimes you don't need plain string, but JSON output.
# for that purpose you use JSON parser which in python will return you a dict.

from langchain_core.output_parsers import JsonOutputParser

output_parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extractor. 
Always respond with valid JSON only. 
No explanation, no markdown, just raw JSON."""),
    ("human", "Extract the name, role, and years of experience from: {text}")
])


chain = prompt | llm | output_parser

result = chain.invoke({
    "text": "Aryan is a backend engineer with 3 years of experience"
})

print(result)  # returns a dict.



# there is a problem with the above method.
# Model sometime return rubbish explanations, garbage texts and markdowns around the json output.
# One possible solution for this is to make your system prompt robust enough to make the model
# only return json outputs.


# A more production ready method is using Pydantic which also allows validations and enforces
# the model to only return object type outputs.
# More on that below...

{'name': 'Aryan', 'role': 'backend engineer', 'years_of_experience': 3}


In [4]:
# Using Pydantic for output parsing.
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Output(BaseModel):
    title: str = Field(description="Short title of the user story")
    description: str = Field(description="As a [user], I want [goal] so that [reason]")
    priority: str = Field(description="high, medium, or low")
    story_points: int = Field(description="Estimated complexity points 1-13")

parser = PydanticOutputParser(pydantic_object= Output) # can be used to parse the output.

print(parser.get_format_instructions()) # can be used to pass the instructions in the system prompt.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "Short title of the user story", "title": "Title", "type": "string"}, "description": {"description": "As a [user], I want [goal] so that [reason]", "title": "Description", "type": "string"}, "priority": {"description": "high, medium, or low", "title": "Priority", "type": "string"}, "story_points": {"description": "Estimated complexity points 1-13", "title": "Story Points", "type": "integer"}}, "required": ["title", "description", "priority", "story_points"]}
```


In [8]:
# you can simply pass this output instruction in your prompt.
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior agile coach.
Break down the given feature into user stories.

{format_instructions}"""),
    ("human", "Feature: {feature}")
])

partial_prompt = prompt.partial(format_instructions = parser.get_format_instructions)

chain = partial_prompt | llm | parser

result = chain.invoke({
    "feature": "Complete user authentication system with login, register, and password reset"
})

print(result)

title='User Login with Credentials' description='As a registered user, I want to log in with my email and password so that I can securely access my account and personal data.' priority='high' story_points=5


In [ ]:
# a more easier method is to use .with_structured_output() method.
from typing import List
class CodeReview(BaseModel):
    has_bugs: bool = Field(description="Whether the code contains bugs")
    bugs: List[str] = Field(description="List of bugs found")
    severity: str = Field(description="critical, high, medium, or low")
    suggestions: List[str] = Field(description="Improvement suggestions")

llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

llm = llm.with_structured_output(CodeReview)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert code reviewer."),
    ("human", "Review this code:\n\n{code}")
])

chain = prompt | llm

result = chain.invoke({
    "code": """
def divide(a, b):
    return a / b
"""
})

True
<class '__main__.CodeReview'>


In [18]:
print(result.has_bugs)
print(type(result))

# converting class into a dict
print(type(result.model_dump()))
print(result.model_dump())

True
<class '__main__.CodeReview'>
<class 'dict'>
{'has_bugs': True, 'bugs': ["The function does not handle division by zero, which will raise a ZeroDivisionError if 'b' is 0."], 'severity': 'medium', 'suggestions': ["Add a check for 'b == 0' and handle it appropriately (e.g., raise a ValueError or return None/float('inf')).", "Add type hints to the function signature, e.g., 'def divide(a: float, b: float) -> float:'.", "Add a docstring to document the function's parameters, return type, and potential exceptions."]}
